In [2]:
"""
ASSIGNMENT INSTRUCTIONS

Expand upon the code for the 3x3 tile sliding puzzle:

https://github.com/everestso/Summer24/blob/main/AI24Ch3a.ipynbLinks to an external site.

to operate on the 4x4 15 puzzle.

Compare and contrast the performance for the uninformed and informed search algorithms on the 3x3 and 4x4 sized puzzle by:

Generate 15 random problems in the 3x3 and 15 random problems in the 4x4  puzzles by generating:

3: Random Walk 5 steps

3: Random Walk 10 steps

3: Random Walk 20 steps

3: Random Walk 40 steps

3: Random Walk 80 steps

For example with the 3x3, generate the problems by starting with state 123456780 and using the random walk function to take random steps to a random state, and then similarly with 4x4.

Test the Breadth-First Search and A* with out-of-place and Manhattan-Distance heurstics.

For each random problem document: (1) start state, (2) the solution, (3) length of solution, and (4) nodes expanded.

Explain what these results indicate about search as a tool for Artificial Intelligence, especially as related to increase complexity in problem size.


Document all results in Google Colab Notebook and save to Github.  Submit link to Github file for assignment.

"""

'\nASSIGNMENT INSTRUCTIONS\n\nExpand upon the code for the 3x3 tile sliding puzzle:\n\nhttps://github.com/everestso/Summer24/blob/main/AI24Ch3a.ipynbLinks to an external site.\n\nto operate on the 4x4 15 puzzle. \n\nCompare and contrast the performance for the uninformed and informed search algorithms on the 3x3 and 4x4 sized puzzle by:\n\nGenerate 15 random problems in the 3x3 and 15 random problems in the 4x4  puzzles by generating:\n\n3: Random Walk 5 steps\n\n3: Random Walk 10 steps\n\n3: Random Walk 20 steps\n\n3: Random Walk 40 steps\n\n3: Random Walk 80 steps\n\nFor example with the 3x3, generate the problems by starting with state 123456780 and using the random walk function to take random steps to a random state, and then similarly with 4x4.\n\nTest the Breadth-First Search and A* with out-of-place and Manhattan-Distance heurstics.\n\nFor each random problem document: (1) start state, (2) the solution, (3) length of solution, and (4) nodes expanded.\n\nExplain what these resul

In [3]:
import random
import heapq

In [40]:
StateDimension=4 # 4 dimensions
InitialState = [1,2,3,4,5,6,7,8,0,9,10,11,12,13,14,15] # new intial state
GoalState=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0]      # new goal state
Actions = lambda s: ['u', 'd', 'l', 'r']
Opposite=dict([('u','d'),('d','u'),('l','r'),('r','l'), (None, None)])

In [41]:
# ALL FUNCTIONS

def Result(state, action):
  i = state.index(0)
  newState = list(state)
  row,col=i//StateDimension, i % StateDimension
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return newState
  if action=='u':
    l,r = row*StateDimension+col, (row-1)*StateDimension+col
  elif action=='d':
    l,r = row*StateDimension+col, (row+1)*StateDimension+col
  elif action=='l':
    l,r = row*StateDimension+col, row*StateDimension+col-1
  elif action=='r' :
    l,r = row*StateDimension+col, row*StateDimension+col+1
  newState[l], newState[r] = newState[r], newState[l]
  return newState

def PrintState(s):
  for i in range(0,len(s),StateDimension):
    print(s[i:i+StateDimension])

def LegalMove(state, action):
  i = state.index(0)
  row,col=i//StateDimension, i % StateDimension
  newState = state.copy()
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return False
  return True

def SingleTileManhattanDistance(tile, left, right):
  leftIndex = left.index(tile)
  rightIndex = right.index(tile)
  return (abs(leftIndex//StateDimension-rightIndex//StateDimension) +
          abs(leftIndex%StateDimension-rightIndex%StateDimension))

def ManhattanDistance(left, right):
  distances = [SingleTileManhattanDistance(tile, left, right)
     for tile in range(1, StateDimension**2)]
   ###  print ("Distances= ", distances)
  return sum(distances)

def OutOfPlace(left, right):
  distances = [left[i]!=right[i] and right[i] != 0
     for i in range(StateDimension**2)]
  return sum(distances)

# random walk - randomizes puzzles
def RandomWalk(state, steps):
  actionSequence = []
  actionLast = None
  for i in range(steps):
    action = None
    while action==None:
      action = random.choice(Actions(state))
      action = action if (LegalMove(state, action)
          and action!= Opposite[actionLast]) else None
    actionLast = action
    state = Result(state, action)
    actionSequence.append(action)
  return state, actionSequence

def ApplyMoves(actions, state):
  for action in actions:
    state = Result(state, action)
  return state

def ReverseMoves(actions):
  ret = [Opposite[a] for a in actions]
  ret.reverse()
  return ret

In [39]:
# ALL SEARCH FUNCTIONS